# DeepSeek OCR - Fine-Tuning Incremental V2

Notebook para **fine-tuning incremental** del modelo DeepSeek-OCR con:
- Dataset español desde HuggingFace (`Lacax/Tickets`): 47 originales + 470 aumentados + 200 sintéticos
- Modelo ya entrenado cargado desde `Lacax/deepseek_ocr_lora`
- **Sin CORD-v2** (ya fue aprendido en el entrenamiento anterior)


### 1. Instalación de dependencias

In [ ]:
# Instalación de dependencias para RunPod
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !rm -f /usr/local/lib/python3.11/dist-packages/typing_extensions.py || true
    !pip install --force-reinstall --no-cache-dir 'typing-extensions>=4.10.0'
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer addict matplotlib


### 2. Cargar modelo base + LoRA existente (incremental)

In [ ]:
from unsloth import FastVisionModel
import torch
from transformers import AutoModel
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'

# Token de HuggingFace (necesario para repos privados)
HF_TOKEN = os.environ.get("HF_TOKEN", "TU_TOKEN_AQUI")  # <-- PON TU TOKEN

# Descargar modelo base DeepSeek-OCR-2
from huggingface_hub import snapshot_download
snapshot_download("unsloth/DeepSeek-OCR-2", local_dir="deepseek_ocr2")

# Cargar modelo base
model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr2",
    load_in_4bit = False,
    auto_model = AutoModel,
    trust_remote_code = True,
    unsloth_force_compile = True,
    use_gradient_checkpointing = "unsloth",
)

# Cargar adaptadores LoRA ya entrenados (fine-tuning incremental)
from peft import PeftModel
model = PeftModel.from_pretrained(
    model,
    "Lacax/deepseek_ocr_lora",
    token=HF_TOKEN,
    is_trainable=True,
)
print("✅ Modelo base + LoRA cargados para fine-tuning incremental")
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


### 3. Cargar dataset español desde HuggingFace (Lacax/Tickets)

Solo datos nuevos — CORD-v2 ya fue aprendido en el entrenamiento anterior.

In [ ]:
import os
import json
from datasets import load_dataset, Dataset
from huggingface_hub import snapshot_download

# Prompt de instrucción para OCR
instruction = """<image>

Extract the following information from the receipt and return it STRICTLY as a valid JSON object matching this structure:

{
  "comercio": "string",
  "cif": "string",
  "fecha": "string",
  "total": "number",
  "items": [{"descripcion": "string", "precio": "number"}]
}

NO other text. ONLY valid JSON.
"""

# Descargar dataset desde HuggingFace
print("Descargando dataset Lacax/Tickets...")
mi_dataset_path = snapshot_download(
    "Lacax/Tickets",
    repo_type="dataset",
    token=HF_TOKEN,
    local_dir="mi_dataset"
)

# Apuntar a la subcarpeta dataset_final/
data_root = os.path.join(mi_dataset_path, "dataset_final")
jsonl_path = os.path.join(data_root, "dataset_espanol.jsonl")
print(f"JSONL: {jsonl_path}")

def format_spanish_ticket(sample):
    full_image_path = os.path.join(data_root, sample["image_path"])
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": full_image_path},
                {"type": "text", "text": instruction}
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["ground_truth"]}
            ]}
        ]
    }

es_dataset_raw = load_dataset("json", data_files=jsonl_path, split="train")
converted_dataset = es_dataset_raw.map(format_spanish_ticket, remove_columns=es_dataset_raw.column_names)
converted_dataset = converted_dataset.shuffle(seed=42)

print(f"\n✅ Dataset cargado: {len(converted_dataset)} tickets")
print(f"   Originales: 47 | Aumentados: 470 | Sintéticos: 200")


### 4. DataCollator (procesamiento de imágenes para DeepSeek)

In [ ]:
# ==============================================================================
# CAMBIO EN ESTA CELDA (Adapta el dataset CORD-v2 al formato JSON Español)
# ==============================================================================
import json
import re

instruction = """<image>

Extract the following information from the receipt and return it STRICTLY as a valid JSON object matching this structure:

{
  "comercio": "string",
  "cif": "string",
  "fecha": "string",
  "total": "number",
  "items": [{"descripcion": "string", "precio": "number"}]
}

NO other text. ONLY valid JSON.
"""

def extract_spanish_json_from_cord(ground_truth_str):
    """
    Parsea la estructura compleja de CORD-v2 y mapea los campos a nuestro
    esquema JSON en español para que el modelo aprenda la estructura correcta.
    """
    try:
        gt_data = json.loads(ground_truth_str)
        gt_parse = gt_data.get("gt_parse", {})
        
        # Extracción segura de la tienda (Comercio y CIF/Tax ID)
        store_info = gt_parse.get("store_name", {})
        comercio = store_info.get("value", "") if isinstance(store_info, dict) else ""
        
        # En CORD-v2 el CIF suele estar en store_tax_info o sub_total/tax
        cif = ""
        
        # Fecha
        fecha = ""
        fecha_info = gt_parse.get("sub_total", {}).get("tax_price", "")
        # Sometimes CORD-v2 masks date, but we can attempt to fetch it if it's in receipt
        # Since cord-v2 doesn't have a standardized date key, we initialize it to empty 
        # to match our CIF strategy or let it learn from your Spanish tickets.

        # Total
        total_info = gt_parse.get("total", {})
        total_val = 0.0
        if isinstance(total_info, dict):
             total_str = total_info.get("total_price", "0")
             try:
                 total_val = float(re.sub(r'[^0-9.]', '', str(total_str)))
             except:
                 pass
                 
        # Items / Menú
        items_out = []
        menu = gt_parse.get("menu", [])
        if isinstance(menu, list):
            for item in menu:
                if isinstance(item, dict):
                    desc = item.get("nm", "")
                    price_str = item.get("price", "0")
                    price_val = 0.0
                    try:
                        price_val = float(re.sub(r'[^0-9.]', '', str(price_str)))
                    except:
                        pass
                    if desc:
                        items_out.append({"descripcion": desc, "precio": price_val})
                        
        return {
            "comercio": comercio,
            "cif": cif,
            "fecha": fecha,
            "total": total_val,
            "items": items_out
        }
    except Exception as e:
        # En caso de error de parseo, devolvemos esquema vacío
        return {
            "comercio": "",
            "cif": "",
            "fecha": "",
            "total": 0.0,
            "items": []
        }

def convert_to_conversation(sample):
    """Convert dataset sample to conversation format for naver-clova-ix/cord-v2"""
    
    # NUEVO: Obtenemos el diccionario adaptado a nuestro esquema JSON español
    mapped_data = extract_spanish_json_from_cord(sample["ground_truth"])
    
    # Formatemos con sangría para mejor visualización (opcional)
    json_output_content = json.dumps(mapped_data, ensure_ascii=False)

    conversation = [
        {
            "role": "<|User|>",
            "content": instruction,
            "images": [sample['image']]
        },
        {
            "role": "<|Assistant|>",
            "content": json_output_content 
        },
    ]
    return {"messages": conversation}

# Re-cargamos el dataset con las conversiones
dataset = load_dataset("naver-clova-ix/cord-v2", split = "train[:400]")
converted_dataset = [convert_to_conversation(sample) for sample in dataset]
print("Ejemplo de dataset adaptado:")
print(converted_dataset[0])


In [ ]:
import torch
import math
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
import io

from deepseek_ocr2.modeling_deepseekocr2 import (
    format_messages,
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

@dataclass
class DeepSeekOCR2DataCollator:
    """
    Args:
        tokenizer: Tokenizer
        model: Model
        image_size: Size for image patches (default: 768)
        base_size: Size for global view (default: 1024)
        crop_mode: Whether to use dynamic cropping for large images
        train_on_responses_only: If True, only train on assistant responses (mask user prompts)
    """
    tokenizer: Any
    model: Any
    image_size: int = 768
    base_size: int = 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(
        self,
        tokenizer,
        model,
        image_size: int = 768,
        base_size: int = 1024,
        crop_mode: bool = True,
        train_on_responses_only: bool = True,
    ):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype  # Get dtype from model
        self.train_on_responses_only = train_on_responses_only

        self.image_transform = BasicImageTransform(
            mean = (0.5, 0.5, 0.5),
            std = (0.5, 0.5, 0.5),
            normalize = True
        )
        self.patch_size = 16
        self.downsample_ratio = 4

        # Get BOS token ID from tokenizer
        if hasattr(tokenizer, 'bos_token_id') and tokenizer.bos_token_id is not None:
            self.bos_id = tokenizer.bos_token_id
        else:
            self.bos_id = 0
            print(f"Warning: tokenizer has no bos_token_id, using default: {self.bos_id}")

    def deserialize_image(self, image_data) -> Image.Image:
        """Convert image data (bytes dict or PIL Image) to PIL Image in RGB mode"""
        if isinstance(image_data, Image.Image):
            img = image_data.convert("RGB")
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            image_bytes = image_data['bytes']
            img = Image.open(io.BytesIO(image_bytes))
            img = img.convert("RGB")
        else:
            raise ValueError(f"Unsupported image format: {type(image_data)}")
        return ImageOps.exif_transpose(img) # Apply EXIF rotation

    def visualize_image(self, image: Image.Image, title: str = ""):
        """Displays the image for visualization purposes."""
        # You can add more sophisticated visualization logic here (e.g., using matplotlib)
        print(f"Displaying image: {title}")
        image.show() # This will attempt to open the image in a viewer

    def calculate_image_token_count(self, image: Image.Image, crop_ratio: Tuple[int, int]) -> int:
        """Calculate the number of tokens this image will generate"""
        num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
        num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

        width_crop_num, height_crop_num = crop_ratio

        if self.crop_mode:
            img_tokens = num_queries_base * num_queries_base + 1
            if width_crop_num > 1 or height_crop_num > 1:
                img_tokens += (num_queries * width_crop_num) * (num_queries * height_crop_num)
        else:
            img_tokens = num_queries * num_queries + 1

        return img_tokens

    def process_image(self, image: Image.Image) -> Tuple[List, List, List, List, Tuple[int, int]]:
        """
        Process a single image based on crop_mode and size thresholds

        Returns:
            Tuple of (images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio)
        """
        images_list = []
        images_crop_list = []
        images_spatial_crop = []

        if self.crop_mode:
            # Determine crop ratio based on image size
            if image.size[0] <= 768 and image.size[1] <= 768:
                crop_ratio = (1, 1)
                images_crop_raw = []
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(
                    image, min_num = 2, max_num = 6,
                    image_size = self.image_size, use_thumbnail = False
                )

            # Process global view with padding
            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color = tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))

            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])

            # Process local views (crops) if applicable
            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(
                        self.image_transform(crop_img).to(self.dtype)
                    )

            # Calculate image tokens
            num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

            tokenized_image = ([self.image_token_id] * num_queries_base) * num_queries_base
            tokenized_image += [self.image_token_id]

            if width_crop_num > 1 or height_crop_num > 1:
                tokenized_image += ([self.image_token_id] * (num_queries * width_crop_num)) * (
                    num_queries * height_crop_num)

        else:  # crop_mode = False
            crop_ratio = (1, 1)
            images_spatial_crop.append([1, 1])

            # For smaller base sizes, resize; for larger, pad
            if self.base_size <= 768:
                resized_image = image.resize((self.base_size, self.base_size), Image.LANCZOS)
                images_list.append(self.image_transform(resized_image).to(self.dtype))
            else:
                global_view = ImageOps.pad(
                    image, (self.base_size, self.base_size),
                    color = tuple(int(x * 255) for x in self.image_transform.mean)
                )
                images_list.append(self.image_transform(global_view).to(self.dtype))

            num_queries = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries) * num_queries
            tokenized_image += [self.image_token_id]

        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    def process_single_sample(self, messages: List[Dict]) -> Dict[str, Any]:
        """
        Process a single conversation into model inputs.
        """

        # --- 1. Setup ---
        images = []
        for message in messages:
            if "images" in message and message["images"]:
                for img_data in message["images"]:
                    if img_data is not None:
                        pil_image = self.deserialize_image(img_data)
                        # Optionally visualize the image after rotation
                        # self.visualize_image(pil_image, title="Processed Image")
                        images.append(pil_image)

        if not images:
            raise ValueError("No images found in sample. Please ensure all samples contain images.")

        tokenized_str = []
        images_seq_mask = []
        images_list, images_crop_list, images_spatial_crop = [], [], []

        prompt_token_count = -1 # Index to start training
        assistant_started = False
        image_idx = 0

        # Add BOS token at the very beginning
        tokenized_str.append(self.bos_id)
        images_seq_mask.append(False)

        for message in messages:
            role = message["role"]
            content = message["content"]

            # Check if this is the assistant's turn
            if role == "<|Assistant|>" :
                if not assistant_started:
                    # This is the split point. All tokens added *so far*
                    # are part of the prompt.
                    prompt_token_count = len(tokenized_str)
                    assistant_started = True

                # Append the EOS token string to the *end* of assistant content
                content = f"{content.strip()} {self.tokenizer.eos_token}"

            # Split this message's content by the image token
            text_splits = content.split('<image>')

            for i, text_sep in enumerate(text_splits):
                # Tokenize the text part
                tokenized_sep = text_encode(self.tokenizer, text_sep, bos = False, eos = False)
                tokenized_str.extend(tokenized_sep)
                images_seq_mask.extend([False] * len(tokenized_sep))

                # If this text is followed by an <image> tag
                if i < len(text_splits) - 1:
                    if image_idx >= len(images):
                        raise ValueError(
                            f"Data mismatch: Found '<image>' token but no corresponding image."
                        )

                    # Process the image
                    image = images[image_idx]
                    img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(image)

                    images_list.extend(img_list)
                    images_crop_list.extend(crop_list)
                    images_spatial_crop.extend(spatial_crop)

                    # Add image placeholder tokens
                    tokenized_str.extend(tok_img)
                    images_seq_mask.extend([True] * len(tok_img))

                    image_idx += 1 # Move to the next image

        # --- 3. Validation and Final Prep ---
        if image_idx != len(images):
            raise ValueError(
                f"Data mismatch: Found {len(images)} images but only {image_idx} '<image>' tokens were used."
            )

        # If we never found an assistant message, we're in a weird state
        # (e.g., user-only prompt). We mask everything.
        if not assistant_started:
            print("Warning: No assistant message found in sample. Masking all tokens.")
            prompt_token_count = len(tokenized_str)

        # Prepare image tensors
        images_ori = torch.stack(images_list, dim = 0)
        images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype = torch.long)

        if images_crop_list:
            images_crop = torch.stack(images_crop_list, dim = 0)
        else:
            images_crop = torch.zeros((1, 3, self.base_size, self.base_size), dtype = self.dtype)

        return {
            "input_ids": torch.tensor(tokenized_str, dtype = torch.long),
            "images_seq_mask": torch.tensor(images_seq_mask, dtype = torch.bool),
            "images_ori": images_ori,
            "images_crop": images_crop,
            "images_spatial_crop": images_spatial_crop_tensor,
            "prompt_token_count": prompt_token_count, # This is now accurate
        }

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        """Collate batch of samples"""
        batch_data = []

        # Process each sample
        for feature in features:
            try:
                processed = self.process_single_sample(feature['messages'])
                batch_data.append(processed)
            except Exception as e:
                print(f"Error processing sample: {e}")
                continue

        if not batch_data:
            raise ValueError("No valid samples in batch")

        # Extract lists
        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]

        # Pad sequences
        input_ids = pad_sequence(input_ids_list, batch_first = True, padding_value = self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first = True, padding_value = False)

        # Create labels
        labels = input_ids.clone()

        # Mask padding tokens
        labels[labels == self.tokenizer.pad_token_id] = -100

        # Mask image tokens (model shouldn't predict these)
        labels[images_seq_mask] = -100

        # Mask user prompt tokens when train_on_responses_only = True (only train on assistant responses)
        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100

        # Create attention mask
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()

        # Prepare images batch (list of tuples)
        images_batch = []
        for item in batch_data:
            images_batch.append((item['images_crop'], item['images_ori']))

        # Stack spatial crop info
        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim = 0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }


### 5. Configurar y lanzar entrenamiento

In [ ]:
from transformers import Trainer, TrainingArguments
from unsloth import is_bf16_supported

FastVisionModel.for_training(model)

data_collator = DeepSeekOCR2DataCollator(
    tokenizer = tokenizer,
    model = model,
    image_size = 768,
    base_size = 1024,
    crop_mode = True,
    train_on_responses_only = True,
)

# LR bajo (1e-5) para no destruir lo ya aprendido
# 2 epochs para evitar overfitting con 717 imágenes
trainer = Trainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = data_collator,
    train_dataset = converted_dataset,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 1e-5,
        logging_steps = 10,
        remove_unused_columns = False,
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        report_to = "none",
        dataloader_num_workers = 4,
        output_dir = "outputs",
    ),
)


In [ ]:
trainer_stats = trainer.train()


### 6. Guardar modelo entrenado

In [ ]:
model.save_pretrained("deepseek_ocr_lora")
tokenizer.save_pretrained("deepseek_ocr_lora")
model.push_to_hub("Lacax/deepseek_ocr_lora", token=HF_TOKEN)
tokenizer.push_to_hub("Lacax/deepseek_ocr_lora", token=HF_TOKEN)
print("✅ Modelo guardado y subido a HuggingFace")


### 7. Test de inferencia

In [ ]:
FastVisionModel.for_inference(model)

from PIL import Image

# Usar una imagen del dataset para probar
test_image_path = os.path.join(data_root, "original", "recibo_almeria_001.jpg")
image = Image.open(test_image_path).convert("RGB")

prompt = """<image>
Extract the following information from the receipt and return it STRICTLY as a valid JSON object matching this structure:

{
  "comercio": "string",
  "cif": "string",
  "fecha": "string",
  "total": "number",
  "items": [{"descripcion": "string", "precio": "number"}]
}

NO other text. ONLY valid JSON.
"""

messages = [{"role": "<|User|>", "content": prompt, "images": [image]},
            {"role": "<|Assistant|>", "content": ""}]

from deepseek_ocr2.modeling_deepseekocr2 import format_messages, text_encode, BasicImageTransform, dynamic_preprocess
text = format_messages(messages, sft_format="deepseek")
input_ids, _ = text_encode(text, tokenizer)

input_ids = torch.tensor([input_ids], device=model.device)
attention_mask = torch.ones_like(input_ids)

# Procesar imagen
transform = BasicImageTransform(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5), normalize=True)
patches = dynamic_preprocess(image, image_size=768, max_num=4)
pixel_values = torch.stack([transform(p) for p in patches]).unsqueeze(0).to(model.device, model.dtype)

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values,
        max_new_tokens=1024,
        do_sample=False,
    )

response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
print("Resultado OCR:")
print(response)
